# EOG / EMG drowsiness detection

The finished detector, shown working. Nothing is trained here: the model comes from `EOG_EDA.ipynb` and the detector code from `drowsiness/bio/eye_events.py`.

* **Part A, pre-recorded data:** a recording is played like a live sensor and the dashboard updates as it runs.
* **Part B, live sensor:** the same dashboard on your EMG sensor connected to the laptop (a simulated sensor is included to try it first).

**What the dashboard shows**

| | |
|---|---|
| **verdict** | `ALERT`, `DROWSY`, `NOT SURE` (signal too poor to say), `LEARNING` (still measuring your normal eye activity) or `NO DATA` |
| **score** | 0 = alert, 1 = drowsy. The verdict says DROWSY from 0.5 |
| **confidence** | 0-1, how much to trust the score: it falls at once when the signal is flat, clipped, noisy or missing |
| **eye** | `closing` while an eye event (blink or long closure) is in progress, with how long it has lasted |
| **events/min, closure share** | how often the eyes close and for what share of the last 2 minutes |
| red marks | the eye events found in the signal (top plot) |

The first **7.5 minutes** (30 s noise learning + 5 min of your normal eye activity + 2 min window) give no score: the detector compares you with your own start. Everything is relative to you, so the sensor's gain and wiring do not matter.

**Honest limits.** The model is trained on only 7 people with self-reported labels: AUC about 0.65 on unseen people, a trend indicator and not a reliable alarm (details in `EOG_EDA.ipynb`). The EMG sensor is a different sensor from the EOG used for training, so the EMG output is **not validated** until it is tested on real data. For EOG/EMG a long closure is not reliable, so this branch must not trigger a microsleep alarm in the fusion.

## Setup

In [1]:
import io, os, time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyedflib
from IPython.display import Image, display

from drowsiness.bio.eye_events import DrowsinessMonitor, LinearModel, SimulatedSource, make_source, verdict
from drowsiness.bio.synthetic import synthetic_emg

MODEL_PATH = "models/eog_drowsiness_model.json"      # made by EOG_EDA.ipynb (section 12)
model = LinearModel.load(MODEL_PATH)
print("model loaded; AUC on unseen people when it was trained:", round(model.info["auc_unseen_people_mean"], 2))

model loaded; AUC on unseen people when it was trained: 0.65


**The dashboard function** (used by both parts). It runs the monitor against the clock: every 2 seconds it redraws the last 20 s of the signal, the verdict and the score history, and returns a table with one row per redraw.

In [2]:
COLOUR = {"ALERT": "tab:green", "DROWSY": "tab:red", "NOT SURE": "tab:orange", "LEARNING": "grey", "NO DATA": "grey"}


def run_dashboard(source, monitor, seconds, speed=1.0, plot_every=2.0, marks=None, show_s=20):
    """Run `monitor` on `source` for `seconds` of sensor time (speed > 1 plays a recording faster than real time).
    marks: optional times (s) of button presses, drawn under the score. Returns a log with one row per redraw."""
    source.start()
    t0, handle, log, next_plot = time.time(), None, [], 0.0
    try:
        while True:
            now = (time.time() - t0) * speed
            if now > seconds:
                break
            times, values = source.read(now)
            out = monitor.update(times, values) if len(times) else monitor.latest(now)
            if now >= next_plot:
                next_plot += plot_every * speed
                raw, word = out["raw"], verdict(out)
                log.append({"t": now, "verdict": word, "score": out["score"], "confidence": out["confidence"],
                            "events_per_min": raw.get("events_per_min", np.nan), "closure_share": raw.get("closure_share", np.nan)})
                lg = pd.DataFrame(log)
                fig, axes = plt.subplots(2, 1, figsize=(10, 5.4)); fig.subplots_adjust(hspace=0.55)
                det = monitor.stream.detector
                tr = np.array(monitor.stream.trace)
                if len(tr):
                    keep = tr[:, 0] > now - show_s
                    axes[0].plot(tr[keep, 0], tr[keep, 1], lw=0.7)
                    for e in det.events:
                        if e["end"] > now - show_s:
                            axes[0].axvspan(e["start"], e["end"], color="r", alpha=0.35)
                    if det.in_event:
                        axes[0].axvspan(det.start_t, now, color="r", alpha=0.6)
                    axes[0].set_xlim(now - show_s, now + 0.5)
                axes[0].set_ylabel("eye activity"); axes[0].set_xlabel("time (s); red = eye closing / blinking", fontsize=8)
                axes[1].fill_between(lg.t, 0, 1, where=(lg.verdict == "DROWSY"), color="tab:red", alpha=0.15, step="mid")
                axes[1].plot(lg.t, lg.score, "k", label="score"); axes[1].plot(lg.t, lg.confidence, color="tab:green", label="confidence")
                axes[1].axhline(0.5, color="r", ls=":", lw=0.8)
                if marks is not None:
                    m = np.asarray(marks); axes[1].vlines(m[m <= now], 0.93, 1.0, color="r", lw=0.8, label="driver pressed 'drowsy'")
                axes[1].set_xlabel("time (s)"); axes[1].set_ylim(-0.05, 1.05); axes[1].set_xlim(0, max(now, 1)); axes[1].legend(loc="lower right", fontsize=7, ncol=3)
                if raw["status"] == "ok":
                    info = f"score {out['score']:.2f}   confidence {out['confidence']:.2f}   events/min {raw['events_per_min']:.0f}   closure share {raw['closure_share']:.0%}   eye {'CLOSING %.1f s' % raw['current_event_s'] if raw['eye_closing_now'] else 'open'}"
                else:
                    info = raw["status"]
                fig.suptitle(f"{now / 60:.1f} min    {word}", color=COLOUR[word], fontsize=16, fontweight="bold", y=1.02)
                axes[0].set_title(info, fontsize=9)
                buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=65, bbox_inches="tight"); plt.close(fig)
                picture = Image(data=buf.getvalue())
                handle = display(picture, display_id=True) if handle is None else (handle.update(picture) or handle)
            time.sleep(0.03)
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        source.stop()
    return pd.DataFrame(log)

# Part A. Pre-recorded data

A recording of the DD-Database (a driver in a night-time simulator who pressed a button whenever they felt drowsy) is played like a sensor, 60 times faster than real time. The red ticks under the score are the real button presses, so you can see whether the verdict follows what the driver reported. Expect a **trend**, not a perfect match (see the limits above).

In [3]:
RECORDING = "03F_1"      # trial 1 of subjects 1-7: 01M_1 02F_1 03F_1 04M_1 05M_1 06M_1 07F_1
MINUTES = 45             # how much of the recording to play
SPEED = 60               # 60 = one minute of recording per second

path = f"data/dd_database/{RECORDING}.edf"
assert os.path.exists(path), "the recording is missing: run EOG_EDA.ipynb first (it downloads the data)"
r = pyedflib.EdfReader(path); labels = r.getSignalLabels()
eye = r.readSignal(labels.index("LOC-Ref")) - r.readSignal(labels.index("ROC-Ref")); r.close()          # the eye signal
a = pyedflib.EdfReader(f"data/dd_database/{RECORDING}_annotations.edf"); presses = np.array(a.readAnnotations()[0]); a.close()
presses = presses[(presses > 30) & (presses < 7000)]
FS = 128

times = np.arange(len(eye)) / FS
monitor = DrowsinessMonitor(model, FS, "eog")
replay_log = run_dashboard(SimulatedSource(times, eye, SPEED), monitor, MINUTES * 60, speed=SPEED, marks=presses)

**What the replay found.** Time spent in each verdict, and the table of the last updates:

In [4]:
print((replay_log.verdict.value_counts() / len(replay_log)).round(2).to_string())
print(f"button presses in the played part: {int((presses < MINUTES * 60).sum())}")
replay_log.round(2).tail(8)

verdict
DROWSY      0.61
ALERT       0.22
LEARNING    0.17
button presses in the played part: 17


,t,verdict,score,confidence,events_per_min,closure_share
15,1801.75,ALERT,0.46,1.0,26.5,0.08
16,1920.15,DROWSY,0.63,1.0,20.5,0.06
17,2041.93,DROWSY,0.78,1.0,15.5,0.05
18,2162.42,DROWSY,0.64,1.0,22.5,0.10
19,2280.10,DROWSY,0.80,1.0,15.5,0.06
20,2401.50,DROWSY,0.63,1.0,20.5,0.07
21,2521.13,DROWSY,0.76,1.0,19.0,0.09
22,2641.28,ALERT,0.40,1.0,28.5,0.08


# Part B. Live sensor

## B1. Try it without a sensor (simulated EMG)
Plays a **synthetic** EMG signal with known blinks: a normal blink rate, then from second 100 only about a fifth of that (the drowsy pattern). It uses short learning times (10 s + 40 s + 30 s) so it fits in a minute, and plays 6 times faster than real time. It proves the live code works; it says nothing about EMG accuracy because the signal is not real.

In [5]:
FS_SIM = 500
sim_t, sim_x, sim_blinks = synthetic_emg(160, FS_SIM, [(0, 16), (100, 3)], mains_hz=50.0, seed=3)
monitor = DrowsinessMonitor(model, FS_SIM, "emg", calibrate_s=10, baseline_s=40, window_s=30)
sim_log = run_dashboard(SimulatedSource(sim_t, sim_x, 6.0), monitor, 160, speed=6.0, plot_every=1.0)

## B2. Your EMG sensor
**Connect, then run the three cells below.**

1. Put the electrodes over the eyelid muscle (orbicularis oculi) at the outer corner of one eye, with the reference on the cheekbone, and connect the sensor to the laptop.
2. The sensor must send **one number per line** over USB serial (for an Arduino: `Serial.println(analogRead(A0));` repeated at a steady rate, for example every 2 ms = 500 Hz) **or** publish a Lab Streaming Layer stream. Close the Arduino Serial Monitor first, as only one program can use the port.
3. Fill in the settings. Everything that depends on your hardware is in this one block.

In [6]:
EMG_SENSOR = {
    "source": "off",               # "serial" (a board printing numbers) or "lsl" (Lab Streaming Layer). "off" = do not run
    "sample_rate_hz": 500,         # samples per second the board really sends; at least 200 (EMG is fast)
    "to_microvolts": 1.0,          # raw number x this = microvolts. ADC counts: volts_per_count x 1e6, divided by the amplifier gain
    "offset": 0.0,                 # raw value at rest, e.g. 512 for a 10-bit ADC with a mid-rail offset
    "polarity": 1,                 # -1 if wired the other way round
    "mains_hz": 50.0,              # India / Europe 50, USA 60
    "port": "/dev/cu.usbmodem1101",    # serial only: pick it from the list printed by the next cell
    "baudrate": 115200,
    "lsl_stream": "EMG",           # lsl only: stream name (or type)
}

In [7]:
from serial.tools import list_ports
print("serial ports on this laptop (the sensor is usually 'usbmodem...' or 'usbserial...'):")
for p in list_ports.comports():
    print("  ", p.device, "-", p.description)

serial ports on this laptop (the sensor is usually 'usbmodem...' or 'usbserial...'):
   /dev/cu.debug-console - n/a
   /dev/cu.Airdopes163 - n/a
   /dev/cu.Bluetooth-Incoming-Port - n/a
   /dev/cu.JBLFlip5 - n/a
   /dev/cu.Stone358 - n/a


**Run the sensor.** The first 7.5 minutes give no score (the detector learns *your* normal eye activity; stay awake and look ahead). Then the verdict appears. Stop it with the stop button of the notebook; the table is saved to `runs/emg_live.csv` (gitignored) when it ends.

In [8]:
if EMG_SENSOR["source"] == "off":
    print('Set EMG_SENSOR["source"] to "serial" or "lsl" in the settings cell, then run this cell again.')
else:
    fs = EMG_SENSOR["sample_rate_hz"]
    monitor = DrowsinessMonitor(model, fs, "emg", EMG_SENSOR["to_microvolts"], EMG_SENSOR["offset"], EMG_SENSOR["polarity"], EMG_SENSOR["mains_hz"])
    live_log = run_dashboard(make_source(EMG_SENSOR), monitor, seconds=3600)
    os.makedirs("runs", exist_ok=True)
    live_log.to_csv("runs/emg_live.csv", index=False)

Set EMG_SENSOR["source"] to "serial" or "lsl" in the settings cell, then run this cell again.


### First session with the sensor: 3 checks (about 10 minutes)
1. **Does it see your blinks?** Blink on command 10 times: `events/min` and the red marks in the top plot should match. If not, check the electrode placement, `to_microvolts`, `offset` and `mains_hz`.
2. **Does bad contact lower the confidence?** Pull an electrode off: the verdict should change to `NOT SURE` within about 2 seconds.
3. **Does the score move the right way?** Hard to test on purpose; record a few real sessions with a way to mark drowsiness (like the dataset's button) and compare. Until then treat the EMG score as unvalidated.